In [ ]:
import numpy as np
import keras_tuner as kt

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# 1. Set a random seed.
keras.utils.set_random_seed(42)


# 2. Load regression data.
X, y = load_diabetes(
    return_X_y=True,
    scaled=False
)


# 3. Create separate training, validation, and test sets.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42
)


# 4. Scale inputs using training statistics only.
x_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)


# 5. Scale regression targets using training statistics only.
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.reshape(-1, 1)
)


# 6. Define a model-building function with tunable settings.
def build_model(hp):
    units = hp.Choice(
        "units",
        values=[16, 32, 64, 128]
    )

    dropout_rate = hp.Choice(
        "dropout_rate",
        values=[0.0, 0.1, 0.2, 0.3]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.001, 0.01]
    )

    model = keras.Sequential([
        keras.Input(shape=(X_train_scaled.shape[1],)),

        layers.Dense(
            units=units,
            activation="relu"
        ),

        layers.Dropout(rate=dropout_rate),

        layers.Dense(1, activation="linear")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )

    return model


# 7. Configure the search.
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective=kt.Objective("val_loss", direction="min"),
    max_trials=10,
    executions_per_trial=1,
    seed=42,
    directory="tuner_results",
    project_name="ann_regression",
    overwrite=True
)

tuner.search_space_summary()


# 8. Search over candidate configurations.
tuner.search(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=100,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)


# 9. Inspect the best trial results.
tuner.results_summary(num_trials=3)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\nBest hyperparameters found:")
print("Hidden units:", best_hp.get("units"))
print("Dropout rate:", best_hp.get("dropout_rate"))
print("Learning rate:", best_hp.get("learning_rate"))


# 10. Build a fresh model using the selected hyperparameters.
best_model = tuner.hypermodel.build(best_hp)


# 11. Train this fresh model.
history = best_model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=200,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )
    ],
    verbose=0
)


# 12. Predict on the held-out test set.
predictions_scaled = best_model.predict(
    X_test_scaled,
    verbose=0
)

predictions = y_scaler.inverse_transform(
    predictions_scaled
).ravel()


# 13. Evaluate in the original target units.
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"\nTest MAE:  {mae:.3f}")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test R²:   {r2:.3f}")